# Ripple Workbench — the bare-bones studio

One notebook. See every table in the Library, pull any slice of it, draw it, put real shadows and glows on it, and export a finished image.

**Runs the same on Windows and Mac.** Nothing here is OS-specific.

### The three tools, and why all three
| Tool | Job here | Why the others can't |
|---|---|---|
| **matplotlib** | draws the chart — bars, axes, labels, in the right places | it's the only one that knows what data *means* |
| **skia-python** | real blur, real drop shadows, real blend modes | matplotlib has no blur; Pillow's is toy-grade |
| **Pillow** | stacks the layers, adds the background, writes the file | the universal adapter everything else speaks |

### First time on a machine
Run this once in a terminal (works on both Mac and Windows — all three ship prebuilt, no compiler needed):
```
pip install matplotlib pillow skia-python pandas
```

### How a notebook works, in one line
Each grey box is a *cell*. Click it, hit `Shift+Enter`, it runs and prints the answer underneath. Run top to bottom the first time.

**The one safety rule:** nothing here can write to Snowflake. Every query goes through the read lane in [viz/sqlrun.py](../viz/sqlrun.py), which refuses anything that isn't a read.

If you wreck a cell: `git checkout playground/workbench.ipynb` puts it back.

---
## 0. Wake it up

Sets the house style and tells you what machine you're on. Run once per session.

In [ ]:
import pathlib, platform, sys

# Find the repo root by walking up until we see CLAUDE.md — works no matter
# where the notebook is opened from, on either OS.
REPO = pathlib.Path.cwd()
while not (REPO / "CLAUDE.md").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
OUT = REPO / "outputs"
OUT.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # draw to memory, never to a window — same on Mac and Windows
import matplotlib.pyplot as plt
from matplotlib import font_manager
import skia
from PIL import Image
from IPython.display import display

# --- the house colours (same ones viz/theme.py uses for the web charts)
BG      = "#0d1117"   # page + chart background
PANEL   = "#161b22"   # panels, hover cards
FG      = "#e6edf3"   # main text
MUTED   = "#8b949e"   # secondary text
GRID    = "#21262d"   # gridlines
ACCENT  = "#3987e5"   # the default blue — series 1, always

# 8 categorical colours in FIXED order. A 9th series folds into "Other";
# never cycle back to blue, or two different things share a colour.
CATEGORICAL = ["#3987e5", "#199e70", "#c98500", "#008300",
               "#9085e9", "#e66767", "#d55181", "#d95926"]

# Blue ramp for magnitude (low -> high). Never a rainbow.
SEQUENTIAL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]


def pick_font():
    """First font on this machine that we actually like. Mac and Windows
    ship different ones, so we ask instead of hardcoding. DejaVu Sans is
    bundled with matplotlib itself, so the last option always exists."""
    have = {f.name for f in font_manager.fontManager.ttflist}
    for name in ("Inter", "SF Pro Text", "Helvetica Neue", "Segoe UI",
                 "Helvetica", "Arial", "DejaVu Sans"):
        if name in have:
            return name
    return "DejaVu Sans"


FONT = pick_font()

plt.rcParams.update({
    "font.family":     FONT,
    "figure.dpi":      110,
    "text.color":      FG,
    "axes.labelcolor": MUTED,
    "xtick.color":     MUTED,
    "ytick.color":     MUTED,
    "axes.edgecolor":  GRID,
    "grid.color":      GRID,
    "axes.facecolor":  "none",     # transparent — Pillow supplies the background later
    "figure.facecolor": "none",
    "axes.grid":       True,
    "grid.linewidth":  0.8,
})

print(f"{platform.system()} {platform.machine()}  |  python {sys.version.split()[0]}")
print(f"font: {FONT}")
print(f"repo: {REPO}")
print(f"matplotlib {matplotlib.__version__} | skia {skia.__version__} | pillow {Image.__version__}")

---
## 1. Connect to the warehouse

Two lanes it can report:
- **enforced** — Snowflake itself refuses writes. Best case.
- **client-guard** — only the Python guard refuses writes. Still safe for you, just less bulletproof.

Skip this section entirely if you just want to play with the drawing tools — sections 4 onward work on any DataFrame, including made-up ones.

In [ ]:
from viz import sqlrun

status = sqlrun.lane_status()
print("lane:      ", status["lane"])
print("warehouse: ", status["warehouse"])
for note in status["notes"]:
    print("  ", note)

---
## 2. What tables exist?

`schemas()` lists the rooms. `tables()` lists what's in one room.

The Library lives in the `THE_LIBRARY` database, which the connection already defaults to — so `GOVERNMENT.SOME_TABLE` works without the full three-part name.

In [ ]:
def q(sql, limit=10_000):
    """Run any read-only SQL. Returns a pandas DataFrame.

    limit — rows to bring back. Default 10k, max 100k.
    """
    df, meta = sqlrun.run(sql, limit_rows=limit)
    flag = "  [TRUNCATED — raise limit= to see more]" if meta["truncated"] else ""
    print(f"{meta['rows']} rows in {meta['elapsed_s']}s{flag}")
    return df


def schemas(database="THE_LIBRARY"):
    """Every room in the database, and how many tables are in it."""
    return q(f"""
        SELECT TABLE_SCHEMA AS schema_name, COUNT(*) AS tables
        FROM {database}.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA <> 'INFORMATION_SCHEMA'
        GROUP BY 1 ORDER BY 2 DESC
    """)


def tables(schema=None, like=None, database="THE_LIBRARY"):
    """Every table, biggest first.

    schema — one room only, e.g. tables("GOVERNMENT")
    like   — fuzzy name search, e.g. tables(like="LOBBY")
    """
    where = ["TABLE_SCHEMA <> 'INFORMATION_SCHEMA'"]
    if schema:
        where.append(f"TABLE_SCHEMA = '{schema.upper()}'")
    if like:
        where.append(f"TABLE_NAME ILIKE '%{like.upper()}%'")
    return q(f"""
        SELECT TABLE_SCHEMA AS schema_name, TABLE_NAME AS table_name,
               ROW_COUNT AS rows, TABLE_TYPE AS kind
        FROM {database}.INFORMATION_SCHEMA.TABLES
        WHERE {' AND '.join(where)}
        ORDER BY ROW_COUNT DESC NULLS LAST
    """)


schemas()

In [ ]:
# Edit and re-run:
tables("GOVERNMENT")
# tables(like="LOBBY")
# tables()                     # everything, biggest first

---
## 3. What's actually inside a table?

| Helper | Answers |
|---|---|
| `columns(t)` | what fields exist, and what type each one is |
| `peek(t)` | what the rows actually look like |
| `keycheck(t, col)` | **is this column real, or fake-populated?** |

`keycheck` is the important one. Twice now a column here has looked 100% full and turned out to be blank strings or placeholder text (NPPES `EIN`, NOAA `imo_number`). Counting non-nulls does not prove a column is real. Counting *distinct* values and eyeballing samples does.

In [ ]:
def columns(table):
    """Field names and types. Pass 'SCHEMA.TABLE'."""
    schema, name = table.upper().split(".")[-2:]
    return q(f"""
        SELECT ORDINAL_POSITION AS pos, COLUMN_NAME AS column_name,
               DATA_TYPE AS type, IS_NULLABLE AS nullable
        FROM THE_LIBRARY.INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = '{schema}' AND TABLE_NAME = '{name}'
        ORDER BY 1
    """)


def peek(table, n=20):
    """The first n rows, so you can see the shape of real values."""
    return q(f"SELECT * FROM {table} LIMIT {n}")


def keycheck(table, column):
    """Is this column trustworthy? Prints the verdict, returns the top values.

    filled   — rows with SOMETHING in it
    distinct — how many DIFFERENT values exist
    Tiny distinct + huge filled = fake-populated: one placeholder repeated.
    """
    r = q(f"""
        SELECT COUNT(*) AS total, COUNT({column}) AS filled,
               COUNT(DISTINCT {column}) AS distinct_vals,
               SUM(CASE WHEN TRIM(TO_VARCHAR({column})) = '' THEN 1 ELSE 0 END) AS blank_strings
        FROM {table}
    """).iloc[0]
    total, filled, distinct = int(r["TOTAL"]), int(r["FILLED"]), int(r["DISTINCT_VALS"])
    blanks = int(r["BLANK_STRINGS"] or 0)
    print(f"\n{table}.{column}")
    print(f"  {filled:,} of {total:,} rows filled ({filled / max(total, 1):.0%})")
    print(f"  {distinct:,} distinct values")
    print(f"  {blanks:,} blank strings hiding as 'filled'")
    ratio = distinct / max(filled, 1)
    if distinct <= 1:
        print("  VERDICT: dead column — one value repeated. Not a key.")
    elif ratio < 0.01:
        print("  VERDICT: suspicious — almost no variety. Read the samples below.")
    elif ratio > 0.9:
        print("  VERDICT: looks like a real identifier.")
    else:
        print("  VERDICT: real data, but repeats — a category, not a unique ID.")
    return q(f"""
        SELECT {column} AS value, COUNT(*) AS n
        FROM {table} WHERE {column} IS NOT NULL
        GROUP BY 1 ORDER BY 2 DESC LIMIT 15
    """)

In [ ]:
TABLE = "GOVERNMENT.CONGRESS_ROLL_CALL_VOTES"   # swap for anything from section 2

columns(TABLE)
# peek(TABLE)
# keycheck(TABLE, "BIOGUIDE_ID")

---
### Pulling your own data

`q()` takes any read-only SQL.

**What gets refused, on purpose:**
- anything that isn't `SELECT` / `WITH` / `SHOW` / `DESCRIBE`
- two statements separated by `;`
- raw reads of the unreviewed claim tables in `LIBRARY_META."CONNECT"` — use `V_LEADS_PUBLISHED`

**SQL gotchas specific to here:**
- Names come back UPPERCASE. `df["FINES"]`, not `df["fines"]`.
- `"CONNECT"` needs its quotes — reserved word.
- Only `THE_LIBRARY` is the default. `LIBRARY_MARTS` / `LIBRARY_RAW` need the full `DATABASE.SCHEMA.TABLE`.
- If you aggregate, every non-aggregated column must be in `GROUP BY`.

In [ ]:
# df = q(f"SELECT * FROM {TABLE} LIMIT 500")
# df.head()

# Once it's a DataFrame it's just Python:
#   df.shape                        (rows, columns)
#   df.columns.tolist()             the field names
#   df["COL"].value_counts()        count each distinct value
#   df.groupby("COL").size()        <- what you need before a bar chart
#   df.sort_values("N", ascending=False).head(20)

# Demo data so everything below runs even with the warehouse asleep:
demo = pd.DataFrame({
    "AGENCY": ["Labor", "Health", "Energy", "Interior", "Transport", "Justice"],
    "FINES":  [412, 388, 190, 155, 96, 74],
    "YEAR":   [2019, 2020, 2021, 2022, 2023, 2024],
})
demo

---
# THE STUDIO

Three stages, three tools, in this order. Each one hands the next a specific thing:

```
  matplotlib          skia               Pillow
  ──────────  →   ──────────   →   ──────────   →  final.png
  draw()          shadow()          stack()
  the chart       the effects       the assembly

  hands over:     hands over:       hands over:
  RGBA array      RGBA array        a PNG file
```

**RGBA array** = a grid of pixels, four numbers each: red, green, blue, and **alpha** (how see-through it is). That's the only handoff format in the whole chain. Every stage takes one and gives one back, which is why they bolt together at all.

**Why this order:** each stage throws something away.
- matplotlib is the last point where things are still *shapes that mean something* ("this is the Labor bar").
- skia is the last point where you can change how pixels *combine*.
- Pillow just stacks flat pictures.

You can't run it backwards. Once matplotlib hands over pixels, "this is a bar" is gone forever.

**The one thing that makes it work:** draw each piece you want to treat separately as its own transparent layer. You can't shadow only the bars if the bars and the gridlines came out in one flat picture.

---
## 4. Stage 1 — matplotlib draws it

`draw()` returns `(fig, ax)`. `render()` turns that into the RGBA array the next stage eats.

**Picking a chart type:**

| Your question | `kind=` | x and y |
|---|---|---|
| Who has the most? | `"bar"` | x = the name, y = the number |
| Who has the most, with long names? | `"barh"` | x = the name, y = the number (it flips them for you) |
| Did it change over time? | `"line"` | x = a date/year, y = the number |
| Do these two move together? | `"scatter"` | both numbers |
| What's the spread? | `"hist"` | x = the number, no y |

**The rule that will bite you:** a bar chart needs *one row per bar*. 500 rows and 12 agencies draws 500 invisible slivers. Aggregate first — `GROUP BY` in SQL or `df.groupby("AGENCY").sum()` in Python.

In [ ]:
def draw(df, kind="bar", x=None, y=None,
         color=None, series=None,
         title=None, subtitle=None, xlabel=None, ylabel=None,
         sort=None, top=None, log_y=False, stack_bars=False,
         size=(9, 5.5), grid_axis="y", source=None):
    """Stage 1: data -> a matplotlib figure. Every knob is listed in section 7.

    Returns (fig, ax). Nothing is drawn to screen yet — use show(fig) or
    render(fig) to see it.
    """
    d = df.copy()
    if sort and sort in d.columns:
        d = d.sort_values(sort, ascending=False)
    if top:
        d = d.head(top)
    if kind == "barh":
        d = d.iloc[::-1]                      # biggest ends up on top

    fig, ax = plt.subplots(figsize=size)
    c = color or ACCENT

    if series:                                # split into one colour per group
        for i, (name, part) in enumerate(d.groupby(series, sort=False)):
            col = CATEGORICAL[i % len(CATEGORICAL)]
            if kind == "line":
                ax.plot(part[x], part[y], color=col, lw=2.4, label=str(name), zorder=3)
            elif kind == "scatter":
                ax.scatter(part[x], part[y], color=col, s=48, label=str(name), zorder=3)
            else:
                ax.bar(part[x], part[y], color=col, width=0.68, label=str(name), zorder=3)
        ax.legend(frameon=False, labelcolor=MUTED)
    else:
        if kind == "bar":
            ax.bar(d[x], d[y], color=c, width=0.68, zorder=3)
        elif kind == "barh":
            ax.barh(d[x], d[y], color=c, height=0.68, zorder=3)
        elif kind == "line":
            ax.plot(d[x], d[y], color=c, lw=2.4, zorder=3)
        elif kind == "scatter":
            ax.scatter(d[x], d[y], color=c, s=48, zorder=3)
        elif kind == "hist":
            ax.hist(d[x], bins=25, color=c, zorder=3)
        else:
            raise ValueError(f"unknown kind {kind!r} — "
                             "use bar / barh / line / scatter / hist")

    if title:
        ax.set_title(title, color=FG, fontsize=17, loc="left",
                     pad=26 if subtitle else 16)
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes,
                color=MUTED, fontsize=11.5, va="bottom")
    ax.set_xlabel(xlabel or "", color=MUTED)
    ax.set_ylabel(ylabel or "", color=MUTED)
    if log_y:
        ax.set_yscale("log")
    if source:
        fig.text(0.01, 0.005, source, color=MUTED, fontsize=9, ha="left")

    ax.grid(axis=grid_axis, zorder=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    fig.tight_layout()
    return fig, ax


def render(fig, dpi=150, close=True):
    """Stage 1 -> Stage 2 handoff. Figure -> transparent RGBA array.

    dpi — how many pixels per inch. 150 is screen-sharp, 300 is print.
          The array gets big fast: doubling dpi quadruples the pixels.
    """
    fig.set_dpi(dpi)
    fig.patch.set_alpha(0.0)
    fig.canvas.draw()
    arr = np.asarray(fig.canvas.buffer_rgba()).copy()
    if close:
        plt.close(fig)
    return arr


def show(x, bg=BG):
    """Look at anything — a figure, an RGBA array, or a Pillow image."""
    if isinstance(x, plt.Figure):
        x = render(x, dpi=110, close=False)
    if isinstance(x, np.ndarray):
        x = Image.alpha_composite(Image.new("RGBA", (x.shape[1], x.shape[0]), bg),
                                  Image.fromarray(x))
    display(x)
    return x

In [ ]:
fig, ax = draw(demo, "barh", x="AGENCY", y="FINES",
               sort="FINES", top=10,
               title="Fines issued, by agency",
               subtitle="Demo data — swap in a real query",
               xlabel="Fines issued",
               source="Source: demo")

art = render(fig)                 # <- the RGBA array everything downstream eats
print("shape:", art.shape, "  (height, width, RGBA)")
show(art)

---
## 5. Stage 2 — skia does the effects

This is the part matplotlib genuinely cannot do. Skia is the graphics engine inside Chrome — it's what actually renders a CSS `box-shadow` or `filter: blur()`.

Each of these takes an RGBA array and gives one back:

| Function | What it makes |
|---|---|
| `shadow(art)` | a **shadow layer on its own** — the chart's silhouette, offset and blurred |
| `glow(art)` | same idea but centred and coloured — a halo |
| `soften(art)` | blurs the thing itself |
| `blend(a, b, mode)` | mathematically combines two layers — multiply, screen, overlay… |

**Why `shadow()` returns the shadow *alone*, not the chart-with-a-shadow:** so Pillow can stack them and you can move, fade or recolour the shadow independently. That's the whole point of the split.

**`pad`** adds transparent space around the edges so a blur has room to spread. Too small and the shadow gets clipped at the border.

In [ ]:
def _to_skia(rgba):
    return skia.Image.fromarray(rgba, colorType=skia.kRGBA_8888_ColorType)


def _from_skia(image):
    return image.toarray(colorType=skia.kRGBA_8888_ColorType)


def _colour(hexstr, opacity=1.0):
    """'#3987e5' + opacity -> the colour value skia wants."""
    h = hexstr.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) / 255 for i in (0, 2, 4))
    return skia.Color4f(r, g, b, opacity).toColor()


def _apply(rgba, image_filter, pad=0):
    """Run one skia filter over an array. Every effect below goes through here."""
    h, w = rgba.shape[:2]
    surface = skia.Surface(w + 2 * pad, h + 2 * pad)
    with surface as canvas:
        canvas.clear(skia.Color4f(0, 0, 0, 0))          # start fully transparent
        paint = skia.Paint(ImageFilter=image_filter)
        canvas.drawImage(_to_skia(rgba), pad, pad, skia.SamplingOptions(), paint)
    return _from_skia(surface.makeImageSnapshot())


def shadow(rgba, dx=6, dy=10, blur=16, colour="#000000", opacity=0.6, pad=48):
    """The shadow layer ALONE. Stack it under the art in stage 3.

    dx, dy   — how far right / down the shadow sits
    blur     — how soft. 0 = hard edge, 40 = a smudge
    colour   — shadows don't have to be black; a dark blue often looks better
    opacity  — 0 invisible, 1 solid
    pad      — transparent margin so the blur isn't clipped. Keep it > blur.
    """
    f = skia.ImageFilters.DropShadowOnly(dx, dy, blur / 2, blur / 2,
                                         _colour(colour, opacity))
    return _apply(rgba, f, pad)


def glow(rgba, blur=24, colour=ACCENT, opacity=0.8, pad=64):
    """A coloured halo — a shadow with no offset. Stack it UNDER the art."""
    f = skia.ImageFilters.DropShadowOnly(0, 0, blur / 2, blur / 2,
                                         _colour(colour, opacity))
    return _apply(rgba, f, pad)


def soften(rgba, radius=8, pad=0):
    """Blur the thing itself. Good for a background layer you want out of focus."""
    return _apply(rgba, skia.ImageFilters.Blur(radius / 2, radius / 2), pad)


def blend(base, top, mode="Multiply", opacity=1.0):
    """Combine two same-size layers mathematically. Pillow cannot do this.

    Modes worth knowing:
      Multiply  — darkens. Like stacking two transparencies.
      Screen    — lightens. Like shining two projectors at one wall.
      Overlay   — boosts contrast: darks darker, lights lighter.
      SoftLight — a gentler Overlay.
      Plus      — straight addition. Blows out fast, good for glows.
    """
    h, w = base.shape[:2]
    surface = skia.Surface(w, h)
    with surface as canvas:
        canvas.clear(skia.Color4f(0, 0, 0, 0))
        canvas.drawImage(_to_skia(base), 0, 0)
        paint = skia.Paint(BlendMode=getattr(skia.BlendMode, "k" + mode),
                           Alphaf=opacity)
        canvas.drawImage(_to_skia(top), 0, 0, skia.SamplingOptions(), paint)
    return _from_skia(surface.makeImageSnapshot())


# See the shadow layer on its own — this is what Pillow will stack underneath.
sh = shadow(art, colour=ACCENT, opacity=0.9, blur=24)
print("shadow layer:", sh.shape, "— note it's bigger than the art, that's the pad")
show(sh)

---
## 6. Stage 3 — Pillow assembles it

Now everything is flat RGBA layers. Pillow's job is boring and reliable: put them in order, put a background behind them, write the file.

`stack()` takes a list of `(layer, x, y)` — **bottom of the pile first**. The x/y is where that layer's top-left corner goes.

Because the shadow layer is bigger than the art (that's the `pad`), the art needs offsetting by the same `pad` to line up. That's the one bit of bookkeeping in the whole pipeline.

In [ ]:
def stack(layers, bg=BG, pad=0):
    """Stage 3: paste layers bottom-first onto a background.

    layers — [(rgba_array, x, y), ...]. First in the list = furthest back.
    bg     — background colour, or "none" for a transparent PNG
    pad    — extra margin around the whole finished image
    """
    W = max(l[0].shape[1] + l[1] for l in layers) + 2 * pad
    H = max(l[0].shape[0] + l[2] for l in layers) + 2 * pad
    canvas = Image.new("RGBA", (W, H), (0, 0, 0, 0) if bg == "none" else bg)
    for rgba, ox, oy in layers:
        canvas.alpha_composite(Image.fromarray(rgba), (ox + pad, oy + pad))
    return canvas


def save(image, name, scale=1.0):
    """Write to outputs/<name>.png. Returns the path."""
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    if scale != 1.0:
        image = image.resize((int(image.width * scale), int(image.height * scale)),
                             Image.LANCZOS)
    path = OUT / f"{name}.png"
    image.save(path)
    print("wrote", path, image.size)
    return path


PAD = 48                                  # must match the pad used in shadow()
final = stack([(sh,  0,   0),             # shadow at the back
               (art, PAD, PAD)],          # art on top, nudged to line up
              bg=BG, pad=24)
display(final)
save(final, "workbench_demo")

---
### The whole pipeline in one cell

This is the shape you'll copy for real work. Six lines of actual pipeline.

In [ ]:
# 1. matplotlib — draw it
fig, ax = draw(demo, "bar", x="AGENCY", y="FINES", sort="FINES",
               title="Fines issued, by agency",
               subtitle="Every stage of the pipeline, one cell")
art = render(fig, dpi=150)

# 2. skia — make the effect layers
glow_layer = glow(art, blur=30, colour=ACCENT, opacity=0.55, pad=64)
drop       = shadow(art, dx=0, dy=14, blur=28, colour="#000000", opacity=0.7, pad=64)

# 3. Pillow — stack and write
poster = stack([(drop, 0, 0), (glow_layer, 0, 0), (art, 64, 64)], bg=BG, pad=32)
display(poster)
save(poster, "workbench_poster")

---
## 7. Every knob, in plain English

### `draw()` — stage 1

**The essentials**
| Knob | Takes | What it does |
|---|---|---|
| `df` | DataFrame | the data. Required. |
| `kind` | `"bar"` `"barh"` `"line"` `"scatter"` `"hist"` | which shape |
| `x` | column name | along the bottom (for `barh`, the labels) |
| `y` | column name | up the side. Leave off for `hist`. |

**Splitting into series**
| Knob | Takes | What it does |
|---|---|---|
| `series` | column name | one coloured series per distinct value — multiple lines, grouped bars |
| `color` | hex string | override the single-series colour. Defaults to the house blue. |

**Trimming**
| Knob | Takes | What it does |
|---|---|---|
| `sort` | column name | order biggest-first by that column |
| `top` | number | keep only N rows *after* sorting. Use it — 200 bars is unreadable. |

**Scale**
| Knob | Takes | What it does |
|---|---|---|
| `log_y` | `True`/`False` | squashes the y-axis so one giant value stops flattening the rest. For magnitudes only — never for counts you want compared fairly. |

**Words**
| Knob | Takes | What it does |
|---|---|---|
| `title` | text | the headline |
| `subtitle` | text | smaller grey line under it |
| `xlabel` / `ylabel` | text | axis labels |
| `source` | text | small grey credit, bottom-left |

**Shape**
| Knob | Takes | What it does |
|---|---|---|
| `size` | `(width, height)` in **inches** | matplotlib thinks in inches; pixels = inches × dpi |
| `grid_axis` | `"y"` `"x"` `"both"` `"none"` | which gridlines show |

---
### `render()` — the stage 1→2 handoff
| Knob | Takes | What it does |
|---|---|---|
| `dpi` | number (150 default) | pixels per inch. 150 screen, 300 print. Doubling it quadruples the file size. |

---
### skia — stage 2
| Function | Knobs |
|---|---|
| `shadow()` | `dx` `dy` (offset), `blur` (softness), `colour`, `opacity`, `pad` |
| `glow()` | `blur`, `colour`, `opacity`, `pad` |
| `soften()` | `radius`, `pad` |
| `blend()` | `mode`, `opacity` |

**`pad` is the one that catches people.** It's transparent margin so the blur has somewhere to spread. Keep `pad` bigger than `blur`, and offset the art by the same `pad` when you stack, or the layers won't line up.

---
### `stack()` and `save()` — stage 3
| Knob | Takes | What it does |
|---|---|---|
| `layers` | `[(array, x, y), ...]` | **first in the list is furthest back** |
| `bg` | hex, or `"none"` | background colour. `"none"` gives a transparent PNG. |
| `pad` | pixels | margin around the whole finished image |
| `scale` | number | resize on save. `0.5` halves it. |

---
### The colour rules (from [viz/theme.py](../viz/theme.py) — same as the web charts)
- **first series is always the same blue**, `ACCENT`
- **8 categorical colours in fixed order** — a 9th series should become "Other", never cycle back to blue
- **one blue ramp for magnitude** (`SEQUENTIAL`), never a rainbow
- **never two y-axes.** Two scales = two charts.

---
## 8. When the knobs aren't enough

Nothing is locked. Two escape hatches:

**matplotlib** — `draw()` hands back the `ax` object. Everything matplotlib can do is on it.
```python
ax.set_ylim(0, 500)
ax.axhline(200, color=MUTED, ls="--", lw=1)      # a reference line
ax.annotate("outlier", xy=(2, 190), color=FG)
ax.tick_params(axis="x", rotation=30)
for bar in ax.patches: bar.set_alpha(0.85)
```

**skia** — `_apply()` takes any skia image filter, not just the four wrapped above. The full list is `dir(skia.ImageFilters)`.
```python
_apply(art, skia.ImageFilters.Dilate(2, 2))       # thicken
_apply(art, skia.ImageFilters.Erode(1, 1))        # thin
```

**To discover what exists:**
```python
[f for f in dir(skia.ImageFilters) if not f.startswith("_")]
[m for m in dir(skia.BlendMode) if m.startswith("k")]
plt.rcParams.keys()                # every matplotlib default, ~1000 of them
```

In [ ]:
fig, ax = draw(demo, "bar", x="AGENCY", y="FINES", title="Tweaked by hand")

ax.axhline(demo["FINES"].mean(), color=MUTED, ls="--", lw=1.2, zorder=4)
ax.text(5.4, demo["FINES"].mean() + 12, "average", color=MUTED, fontsize=10, ha="right")
ax.tick_params(axis="x", rotation=20)
for bar in ax.patches[:2]:
    bar.set_color(CATEGORICAL[2])          # highlight the top two

show(fig)

In [ ]:
# What's available, printed rather than guessed:
print("SKIA FILTERS:")
print(" ", ", ".join(f for f in dir(skia.ImageFilters) if not f.startswith("_"))[:600])
print("\nBLEND MODES:")
print(" ", ", ".join(m[1:] for m in dir(skia.BlendMode) if m.startswith("k")))

---
## 9. When it breaks

| What you see | What it means | Fix |
|---|---|---|
| `resource monitor ... exceeded its quota` | **the Snowflake spend cap is hit — the warehouse won't start for anyone.** No code fixes this. | raise the quota on `SERVE_MON` or wait for the monthly reset |
| `does not exist or not authorized` | wrong name, or your role can't see it | check spelling in section 2; use the full `DATABASE.SCHEMA.TABLE` outside `THE_LIBRARY` |
| `GuardError: ...refused` | that isn't a plain read | rewrite as `SELECT`; for claim tables use `V_LEADS_PUBLISHED` |
| `SQL compilation error ... GROUP BY` | aggregated some columns, not others | every column not inside `COUNT()`/`SUM()` must be in `GROUP BY` |
| `KeyError: 'FINES'` | column name mismatch | `df.columns.tolist()` — Snowflake returns UPPERCASE |
| chart is an empty box / one flat line | you charted raw rows, not totals | aggregate first: `GROUP BY`, or `df.groupby("COL").sum()` |
| shadow is cut off at the edge | `pad` is smaller than `blur` | raise `pad` — and raise the art's offset in `stack()` to match |
| layers are offset / misaligned | the art offset doesn't match the shadow's `pad` | they must be the same number |
| `ValueError: operands could not be broadcast` | `blend()` got two different-sized layers | blend only works on same-size arrays. Pad them first. |
| a blank window appears / nothing shows | wrong matplotlib backend | section 0 sets `Agg` — re-run it |
| fonts look wrong on one machine | that machine doesn't have the font | expected. `pick_font()` picks the best one present; `DejaVu Sans` always exists. |
| `[TRUNCATED]` in the row count | hit the row limit | `q(sql, limit=50_000)` — max 100,000 |
| everything hangs | warehouse waking up | wait ~10s, re-run. First query of the day is always slow. |

**Nuclear reset:** VS Code → `Restart Kernel`, then run section 0 again. Nothing is lost.

**Moving to the Mac:** `git pull`, then `pip install matplotlib pillow skia-python pandas`. All three have prebuilt Apple Silicon wheels — nothing compiles. The `.env` with your Snowflake credentials lives in `library-onboarding/` and is **not** in git, so you'll need to copy that across separately.